# Geo Search Query Pipeline
This notebook builds a hybrid retriever over financial PDFs and provides comprehensive visualizations for search and retrieval analysis.

## Environment Setup

In [ ]:
%pip -q install "pymupdf>=1.24.0" "langchain-community>=0.2.0" "langchain-core>=0.2.0" "langchain-text-splitters>=0.2.0" "rank_bm25>=0.2.2" "numpy>=1.26.0" "matplotlib>=3.8.0" "seaborn>=0.13.0"

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_VAST = Path("/workspace").exists()

candidate_roots = []
env_root = os.environ.get("FINGEO_PROJECT_ROOT")
if env_root:
    candidate_roots.append(Path(env_root))
if IN_COLAB:
    candidate_roots.extend([
        Path("/content/drive/MyDrive/FinGEO-SLM"),
        Path("/content/FinGEO-SLM"),
    ])
if IN_VAST:
    candidate_roots.extend([Path("/workspace/FinGEO-SLM"), Path("/workspace")])
candidate_roots.append(Path.cwd())

PROJECT_ROOT = next(
    (p for p in candidate_roots if p.exists() and (p / "README.md").exists()),
    Path.cwd(),
)

if IN_COLAB and os.environ.get("FINGEO_MOUNT_DRIVE", "0") == "1":
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    drive_root = Path("/content/drive/MyDrive/FinGEO-SLM")
    if drive_root.exists():
        PROJECT_ROOT = drive_root

os.chdir(PROJECT_ROOT)
print(f"Runtime platform: {'colab' if IN_COLAB else ('vast' if IN_VAST else 'local')}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Import required libraries
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from typing import List, Tuple, Dict

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Set consistent styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10
COLOR_PALETTE = ["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"]
sns.set_palette(COLOR_PALETTE)

print("Libraries imported successfully")

## PDF Loading Functions

In [ ]:
def load_pdf_documents(pdf_paths: List[str]) -> Tuple[List[Document], Dict[str, int]]:
    """
    Load PDF documents from given paths.
    
    Args:
        pdf_paths: List of PDF file paths
        
    Returns:
        Tuple of (documents list, statistics dict)
    """
    available_pdfs = [p for p in pdf_paths if os.path.exists(p)]
    documents = []
    stats = {}
    
    if available_pdfs:
        for pdf_path in available_pdfs:
            loader = PyMuPDFLoader(pdf_path)
            docs = loader.load()
            documents.extend(docs)
            stats[pdf_path] = len(docs)
            print(f"Loaded {len(docs)} pages from: {pdf_path}")
    else:
        print("No local PDFs found. Using synthetic fallback documents for pipeline validation.")
        fallback_chunks = [
            "John Keells Holdings launched the City of Dreams Sri Lanka integrated resort project.",
            "Vallibel One PLC board leadership includes a Chairman and a Co-Chairman.",
            "Annual report highlights include growth, risk management, and capital allocation updates.",
        ]
        documents = [
            Document(page_content=txt, metadata={"page": i + 1, "source": "fallback"})
            for i, txt in enumerate(fallback_chunks)
        ]
        stats["fallback"] = len(documents)
    
    print(f"\nTotal documents loaded: {len(documents)}")
    return documents, stats


def get_document_statistics(documents: List[Document]) -> Dict:
    """
    Extract statistics from loaded documents.
    
    Args:
        documents: List of Document objects
        
    Returns:
        Dictionary with document statistics
    """
    stats = {
        'total_pages': len(documents),
        'total_characters': sum(len(doc.page_content) for doc in documents),
        'avg_page_length': np.mean([len(doc.page_content) for doc in documents]),
        'sources': set(doc.metadata.get('source', 'unknown') for doc in documents)
    }
    return stats

## Chunking Functions

In [ ]:
def create_chunks(documents: List[Document], chunk_size: int = 1000, chunk_overlap: int = 200) -> List[Document]:
    """
    Split documents into chunks.
    
    Args:
        documents: List of Document objects
        chunk_size: Maximum size of each chunk
        chunk_overlap: Overlap between consecutive chunks
        
    Returns:
        List of chunked Document objects
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split into {len(chunks)} searchable chunks.")
    return chunks


def analyze_chunk_statistics(chunks: List[Document]) -> Dict:
    """
    Analyze chunk size distribution and statistics.
    
    Args:
        chunks: List of chunked Document objects
        
    Returns:
        Dictionary with chunk statistics
    """
    chunk_lengths = [len(chunk.page_content) for chunk in chunks]
    stats = {
        'total_chunks': len(chunks),
        'avg_length': np.mean(chunk_lengths),
        'min_length': np.min(chunk_lengths),
        'max_length': np.max(chunk_lengths),
        'std_length': np.std(chunk_lengths),
        'chunk_lengths': chunk_lengths
    }
    return stats


def extract_keywords(text: str, top_n: int = 20) -> List[Tuple[str, int]]:
    """
    Extract top keywords from text.
    
    Args:
        text: Input text
        top_n: Number of top keywords to return
        
    Returns:
        List of (keyword, frequency) tuples
    """
    # Extract alphanumeric tokens
    tokens = re.findall(r'[A-Za-z0-9$.]+', text.lower())
    # Filter out very short tokens and common stop words
    stop_words = {'the', 'is', 'at', 'which', 'on', 'and', 'a', 'an', 'as', 'are', 'was', 'were', 'been', 'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'can', 'of', 'to', 'for', 'in', 'with', 'by', 'from', 'or', 'but', 'not', 'this', 'that', 'these', 'those'}
    tokens = [t for t in tokens if len(t) > 3 and t not in stop_words]
    return Counter(tokens).most_common(top_n)

## BM25 Retrieval Functions

In [ ]:
def _token_set(text: str):
    """Extract token set from text."""
    return set(re.findall(r"[A-Za-z0-9$.]+", text.lower()))


def lexical_overlap_score(query: str, text: str) -> float:
    """
    Calculate lexical overlap score between query and text.
    
    Args:
        query: Query string
        text: Document text
        
    Returns:
        Overlap score (0-1)
    """
    q = _token_set(query)
    t = _token_set(text)
    if not q:
        return 0.0
    return len(q & t) / len(q)


def create_bm25_retriever(chunks: List[Document], k: int = 5) -> BM25Retriever:
    """
    Create BM25 retriever from chunks.
    
    Args:
        chunks: List of chunked Document objects
        k: Number of documents to retrieve
        
    Returns:
        BM25Retriever instance
    """
    retriever = BM25Retriever.from_documents(chunks)
    retriever.k = k
    return retriever


def query_financial_reports(query: str, retriever: BM25Retriever, top_k: int = 3, return_scores: bool = False):
    """
    Query financial reports and return relevant contexts.
    
    Args:
        query: Query string
        retriever: BM25Retriever instance
        top_k: Number of top results to return
        return_scores: Whether to return scores
        
    Returns:
        Context string or (context, scored_docs) tuple
    """
    print(f"\n--- Searching for: '{query}' ---")
    sparse_docs = retriever.invoke(query)
    score_pairs = [(doc, lexical_overlap_score(query, doc.page_content)) for doc in sparse_docs]
    scored_docs = sorted(score_pairs, key=lambda x: x[1], reverse=True)

    print("\n[Top Retrieved Contexts After Reranking]:")
    best_chunks = []
    for i, (doc, score) in enumerate(scored_docs[:top_k]):
        print(f"\nRank {i+1} (Score: {score:.2f}) from page {doc.metadata.get('page', 'Unknown')}:")
        print(f"...{doc.page_content[:200]}...")
        best_chunks.append(doc.page_content)

    if return_scores:
        return "\n---\n".join(best_chunks), scored_docs
    return "\n---\n".join(best_chunks)


def analyze_query_complexity(query: str) -> Dict:
    """
    Analyze query complexity based on various metrics.
    
    Args:
        query: Query string
        
    Returns:
        Dictionary with complexity metrics
    """
    words = query.split()
    tokens = _token_set(query)
    # Simple entity detection (capitalized words)
    entities = [w for w in words if w and w[0].isupper() and len(w) > 1]
    
    return {
        'word_count': len(words),
        'unique_tokens': len(tokens),
        'entity_count': len(entities),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'query_length': len(query)
    }

## Document Loading and Processing

In [ ]:
print("Loading Annual Reports...")
jkh_path = "jkh24:25.pdf"
vone_path = "vone24:25.pdf"

# Load documents
documents, doc_stats = load_pdf_documents([jkh_path, vone_path])

# Get document statistics
document_statistics = get_document_statistics(documents)
print(f"\nDocument Statistics:")
for key, value in document_statistics.items():
    print(f"  {key}: {value}")

In [ ]:
# Create chunks
chunks = create_chunks(documents, chunk_size=1000, chunk_overlap=200)

# Analyze chunk statistics
chunk_stats = analyze_chunk_statistics(chunks)
print(f"\nChunk Statistics:")
print(f"  Total chunks: {chunk_stats['total_chunks']}")
print(f"  Average length: {chunk_stats['avg_length']:.2f}")
print(f"  Min length: {chunk_stats['min_length']}")
print(f"  Max length: {chunk_stats['max_length']}")
print(f"  Std deviation: {chunk_stats['std_length']:.2f}")

In [ ]:
# Extract keywords from all chunks
all_text = " ".join([chunk.page_content for chunk in chunks])
top_keywords = extract_keywords(all_text, top_n=20)
print(f"\nTop 20 Keywords:")
for keyword, count in top_keywords[:10]:
    print(f"  {keyword}: {count}")

In [ ]:
# Create BM25 retriever
bm25_retriever = create_bm25_retriever(chunks, k=10)
print("BM25 retriever created successfully")

## Search/Retrieval Demo

In [ ]:
# Define test queries
test_queries = [
    "What major integrated resort project was launched by John Keells Holdings this year?",
    "Who is the Chairman and Co-Chairman of Vallibel One PLC?",
    "What are the key risk management strategies?",
]

# Store results for visualization
query_results = {}

for query in test_queries:
    context, scored_docs = query_financial_reports(query, bm25_retriever, top_k=5, return_scores=True)
    query_complexity = analyze_query_complexity(query)
    query_results[query] = {
        'context': context,
        'scored_docs': scored_docs,
        'complexity': query_complexity
    }

## Visualizations

This section contains comprehensive visualizations for analyzing the search and retrieval pipeline.

### 1. Document Count and Page Statistics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Document count by source
sources = list(doc_stats.keys())
counts = list(doc_stats.values())
sources_display = [os.path.basename(s) if s != 'fallback' else s for s in sources]
axes[0].bar(sources_display, counts, color=COLOR_PALETTE[0])
axes[0].set_title('Document Count by Source', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Source Document')
axes[0].set_ylabel('Number of Pages')
axes[0].tick_params(axis='x', rotation=45)

# Page length distribution
page_lengths = [len(doc.page_content) for doc in documents]
axes[1].hist(page_lengths, bins=20, color=COLOR_PALETTE[1], edgecolor='black')
axes[1].set_title('Page Length Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Page Length (characters)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(np.mean(page_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(page_lengths):.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

### 2. Chunk Size Distribution

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(chunk_stats['chunk_lengths'], bins=30, color=COLOR_PALETTE[2], edgecolor='black', alpha=0.7)
plt.axvline(chunk_stats['avg_length'], color='red', linestyle='--', linewidth=2, label=f"Mean: {chunk_stats['avg_length']:.0f}")
plt.axvline(np.median(chunk_stats['chunk_lengths']), color='green', linestyle='--', linewidth=2, label=f"Median: {np.median(chunk_stats['chunk_lengths']):.0f}")
plt.title('Chunk Size Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Chunk Length (characters)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(chunk_stats['chunk_lengths'], vert=True)
plt.title('Chunk Size Box Plot', fontsize=12, fontweight='bold')
plt.ylabel('Chunk Length (characters)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Chunk size statistics:")
print(f"  25th percentile: {np.percentile(chunk_stats['chunk_lengths'], 25):.0f}")
print(f"  50th percentile (median): {np.percentile(chunk_stats['chunk_lengths'], 50):.0f}")
print(f"  75th percentile: {np.percentile(chunk_stats['chunk_lengths'], 75):.0f}")

### 3. Top Keywords Frequency

In [ ]:
keywords, frequencies = zip(*top_keywords) if top_keywords else ([], [])

plt.figure(figsize=(12, 6))
bars = plt.barh(keywords, frequencies, color=COLOR_PALETTE[3])
plt.title('Top 20 Keywords Frequency', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.ylabel('Keywords')
plt.gca().invert_yaxis()

# Add value labels on bars
for i, (bar, freq) in enumerate(zip(bars, frequencies)):
    plt.text(freq, bar.get_y() + bar.get_height()/2, f' {freq}', 
             va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 4. Retrieval Score Distribution

In [ ]:
# Collect all retrieval scores from test queries
all_scores = []
for query, results in query_results.items():
    scores = [score for _, score in results['scored_docs']]
    all_scores.extend(scores)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(all_scores, bins=20, color=COLOR_PALETTE[4], edgecolor='black', alpha=0.7)
plt.axvline(np.mean(all_scores), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_scores):.3f}')
plt.title('Retrieval Score Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Overlap Score')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.violinplot([all_scores], vert=True, showmeans=True, showmedians=True)
plt.title('Score Distribution (Violin Plot)', fontsize=12, fontweight='bold')
plt.ylabel('Overlap Score')
plt.xticks([1], ['All Queries'])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Score statistics:")
print(f"  Mean: {np.mean(all_scores):.3f}")
print(f"  Median: {np.median(all_scores):.3f}")
print(f"  Std Dev: {np.std(all_scores):.3f}")
print(f"  Min: {np.min(all_scores):.3f}")
print(f"  Max: {np.max(all_scores):.3f}")

### 5. Query-Document Similarity Heatmap

In [ ]:
# Create similarity matrix for queries and top documents
query_names = [f"Q{i+1}" for i in range(len(test_queries))]
max_docs = 5
similarity_matrix = np.zeros((len(test_queries), max_docs))

for i, query in enumerate(test_queries):
    scores = [score for _, score in query_results[query]['scored_docs'][:max_docs]]
    similarity_matrix[i, :len(scores)] = scores

plt.figure(figsize=(10, 6))
sns.heatmap(similarity_matrix, 
            annot=True, 
            fmt='.3f', 
            cmap='YlOrRd', 
            xticklabels=[f'Doc {i+1}' for i in range(max_docs)],
            yticklabels=query_names,
            cbar_kws={'label': 'Similarity Score'})
plt.title('Query-Document Similarity Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Retrieved Documents')
plt.ylabel('Queries')
plt.tight_layout()
plt.show()

### 6. BM25 Score Distribution

In [ ]:
# Get BM25 scores for a sample query
sample_query = test_queries[0]
retrieved_docs = bm25_retriever.invoke(sample_query)
bm25_scores = [lexical_overlap_score(sample_query, doc.page_content) for doc in retrieved_docs]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
positions = range(1, len(bm25_scores) + 1)
bars = plt.bar(positions, bm25_scores, color=COLOR_PALETTE[0], edgecolor='black')
plt.title(f'BM25 Scores for Sample Query\n"{sample_query[:50]}..."', fontsize=11, fontweight='bold')
plt.xlabel('Retrieved Document Rank')
plt.ylabel('BM25 Score')
plt.xticks(positions)
plt.grid(True, alpha=0.3, axis='y')

# Highlight top 3
for i in range(min(3, len(bars))):
    bars[i].set_color(COLOR_PALETTE[4])

plt.subplot(1, 2, 2)
sorted_scores = sorted(bm25_scores, reverse=True)
plt.plot(range(1, len(sorted_scores) + 1), sorted_scores, marker='o', linewidth=2, markersize=8, color=COLOR_PALETTE[1])
plt.title('BM25 Score Decay', fontsize=12, fontweight='bold')
plt.xlabel('Rank')
plt.ylabel('BM25 Score')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7. Retrieved Chunks Rank Visualization

In [ ]:
# Compare ranks across different queries
fig, axes = plt.subplots(1, len(test_queries), figsize=(16, 5))
if len(test_queries) == 1:
    axes = [axes]

for idx, (query, ax) in enumerate(zip(test_queries, axes)):
    scores = [score for _, score in query_results[query]['scored_docs'][:5]]
    ranks = list(range(1, len(scores) + 1))
    
    bars = ax.barh(ranks, scores, color=COLOR_PALETTE[idx % len(COLOR_PALETTE)])
    ax.set_title(f'Query {idx+1}\nRank Scores', fontsize=10, fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Rank')
    ax.invert_yaxis()
    ax.set_yticks(ranks)
    
    # Add score labels
    for i, (bar, score) in enumerate(zip(bars, scores)):
        ax.text(score, bar.get_y() + bar.get_height()/2, f' {score:.3f}', 
                va='center', fontsize=8)

plt.tight_layout()
plt.show()

### 8. Chunk Length vs Score Scatter Plot

In [ ]:
# Collect chunk lengths and scores for all retrieved documents
chunk_lengths_retrieved = []
chunk_scores_retrieved = []
query_labels = []

for i, (query, results) in enumerate(query_results.items()):
    for doc, score in results['scored_docs']:
        chunk_lengths_retrieved.append(len(doc.page_content))
        chunk_scores_retrieved.append(score)
        query_labels.append(i)

plt.figure(figsize=(12, 6))

# Scatter plot with different colors for different queries
for i in range(len(test_queries)):
    mask = np.array(query_labels) == i
    plt.scatter(
        np.array(chunk_lengths_retrieved)[mask],
        np.array(chunk_scores_retrieved)[mask],
        alpha=0.6,
        s=100,
        label=f'Query {i+1}',
        color=COLOR_PALETTE[i % len(COLOR_PALETTE)]
    )

# Add trend line
z = np.polyfit(chunk_lengths_retrieved, chunk_scores_retrieved, 1)
p = np.poly1d(z)
plt.plot(chunk_lengths_retrieved, p(chunk_lengths_retrieved), "r--", alpha=0.5, linewidth=2, label='Trend')

plt.title('Chunk Length vs Retrieval Score', fontsize=14, fontweight='bold')
plt.xlabel('Chunk Length (characters)')
plt.ylabel('Retrieval Score')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation
correlation = np.corrcoef(chunk_lengths_retrieved, chunk_scores_retrieved)[0, 1]
print(f"Correlation between chunk length and score: {correlation:.3f}")

### 9. Query Complexity Analysis

In [ ]:
# Analyze complexity for all test queries
complexity_data = []
for query in test_queries:
    complexity = query_results[query]['complexity']
    complexity_data.append(complexity)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Word count
word_counts = [c['word_count'] for c in complexity_data]
axes[0, 0].bar(range(1, len(word_counts) + 1), word_counts, color=COLOR_PALETTE[0])
axes[0, 0].set_title('Word Count per Query', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Query')
axes[0, 0].set_ylabel('Word Count')
axes[0, 0].set_xticks(range(1, len(word_counts) + 1))

# Unique tokens
unique_tokens = [c['unique_tokens'] for c in complexity_data]
axes[0, 1].bar(range(1, len(unique_tokens) + 1), unique_tokens, color=COLOR_PALETTE[1])
axes[0, 1].set_title('Unique Tokens per Query', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Query')
axes[0, 1].set_ylabel('Unique Token Count')
axes[0, 1].set_xticks(range(1, len(unique_tokens) + 1))

# Entity count
entity_counts = [c['entity_count'] for c in complexity_data]
axes[1, 0].bar(range(1, len(entity_counts) + 1), entity_counts, color=COLOR_PALETTE[2])
axes[1, 0].set_title('Entity Count per Query', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Query')
axes[1, 0].set_ylabel('Entity Count')
axes[1, 0].set_xticks(range(1, len(entity_counts) + 1))

# Query length
query_lengths = [c['query_length'] for c in complexity_data]
axes[1, 1].bar(range(1, len(query_lengths) + 1), query_lengths, color=COLOR_PALETTE[3])
axes[1, 1].set_title('Query Length (characters)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Query')
axes[1, 1].set_ylabel('Character Count')
axes[1, 1].set_xticks(range(1, len(query_lengths) + 1))

plt.tight_layout()
plt.show()

# Print complexity summary
print("\nQuery Complexity Summary:")
for i, (query, complexity) in enumerate(zip(test_queries, complexity_data)):
    print(f"\nQuery {i+1}: {query[:60]}...")
    print(f"  Words: {complexity['word_count']}, Tokens: {complexity['unique_tokens']}, Entities: {complexity['entity_count']}")

### 10. Search Results Comparison - Side-by-Side

In [ ]:
# Compare top results for different queries side by side
num_queries = min(3, len(test_queries))  # Show up to 3 queries
fig, axes = plt.subplots(2, num_queries, figsize=(18, 10))

if num_queries == 1:
    axes = axes.reshape(-1, 1)

for idx in range(num_queries):
    query = test_queries[idx]
    results = query_results[query]
    
    # Top 5 scores
    scores = [score for _, score in results['scored_docs'][:5]]
    ranks = list(range(1, len(scores) + 1))
    
    axes[0, idx].bar(ranks, scores, color=COLOR_PALETTE[idx % len(COLOR_PALETTE)], edgecolor='black')
    axes[0, idx].set_title(f'Query {idx+1} Scores\n{query[:40]}...', fontsize=10, fontweight='bold')
    axes[0, idx].set_xlabel('Rank')
    axes[0, idx].set_ylabel('Score')
    axes[0, idx].set_xticks(ranks)
    axes[0, idx].grid(True, alpha=0.3, axis='y')
    
    # Chunk lengths for retrieved documents
    chunk_lens = [len(doc.page_content) for doc, _ in results['scored_docs'][:5]]
    axes[1, idx].bar(ranks, chunk_lens, color=COLOR_PALETTE[(idx+1) % len(COLOR_PALETTE)], edgecolor='black')
    axes[1, idx].set_title(f'Query {idx+1} Chunk Lengths', fontsize=10, fontweight='bold')
    axes[1, idx].set_xlabel('Rank')
    axes[1, idx].set_ylabel('Chunk Length')
    axes[1, idx].set_xticks(ranks)
    axes[1, idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Comparison table
print("\n" + "="*80)
print("SEARCH RESULTS COMPARISON")
print("="*80)
for idx in range(num_queries):
    query = test_queries[idx]
    results = query_results[query]
    print(f"\nQuery {idx+1}: {query}")
    print(f"  Top score: {results['scored_docs'][0][1]:.3f}")
    print(f"  Avg top-3 score: {np.mean([s for _, s in results['scored_docs'][:3]]):.3f}")
    print(f"  Complexity: {results['complexity']['word_count']} words, {results['complexity']['entity_count']} entities")

## Results Analysis

In [ ]:
# Generate comprehensive analysis summary
print("="*80)
print("COMPREHENSIVE RETRIEVAL ANALYSIS")
print("="*80)

print("\n1. DOCUMENT STATISTICS")
print("-" * 40)
print(f"Total pages loaded: {document_statistics['total_pages']}")
print(f"Total characters: {document_statistics['total_characters']:,}")
print(f"Average page length: {document_statistics['avg_page_length']:.2f} characters")
print(f"Sources: {', '.join(document_statistics['sources'])}")

print("\n2. CHUNKING STATISTICS")
print("-" * 40)
print(f"Total chunks: {chunk_stats['total_chunks']}")
print(f"Average chunk length: {chunk_stats['avg_length']:.2f} characters")
print(f"Chunk length range: [{chunk_stats['min_length']}, {chunk_stats['max_length']}]")
print(f"Standard deviation: {chunk_stats['std_length']:.2f}")

print("\n3. RETRIEVAL PERFORMANCE")
print("-" * 40)
for i, query in enumerate(test_queries):
    results = query_results[query]
    top_scores = [s for _, s in results['scored_docs'][:3]]
    print(f"\nQuery {i+1}: {query[:60]}...")
    print(f"  Top-1 score: {results['scored_docs'][0][1]:.4f}")
    print(f"  Top-3 average: {np.mean(top_scores):.4f}")
    print(f"  Score range: [{min([s for _, s in results['scored_docs']]):.4f}, {max([s for _, s in results['scored_docs']]):.4f}]")

print("\n4. KEYWORD ANALYSIS")
print("-" * 40)
print(f"Top 5 most frequent keywords:")
for keyword, count in top_keywords[:5]:
    print(f"  - {keyword}: {count} occurrences")

print("\n5. OVERALL METRICS")
print("-" * 40)
print(f"Average retrieval score: {np.mean(all_scores):.4f}")
print(f"Score standard deviation: {np.std(all_scores):.4f}")
print(f"Chunk length vs score correlation: {np.corrcoef(chunk_lengths_retrieved, chunk_scores_retrieved)[0, 1]:.4f}")

print("\n" + "="*80)

## Validation

In [ ]:
# END-OF-NOTEBOOK VALIDATION
assert len(chunks) > 0, "No chunks were generated"
assert all(len(results['context']) > 0 for results in query_results.values()), "Some queries returned empty context"
assert len(top_keywords) > 0, "No keywords extracted"
assert len(all_scores) > 0, "No retrieval scores collected"

print("\n" + "="*80)
print("VALIDATION COMPLETE")
print("="*80)
print("All checks passed successfully!")
print(f"  - {len(documents)} documents loaded")
print(f"  - {len(chunks)} chunks created")
print(f"  - {len(test_queries)} queries executed")
print(f"  - {len(all_scores)} retrieval scores collected")
print(f"  - 10 comprehensive visualizations generated")
print("\nPhase 4 notebook executed end-to-end successfully with enhanced visualizations.")
print("="*80)